# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AviJxn/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import os, sys, subprocess
import pandas as pd
import numpy as np

# 1. Environment & Path Setup
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
        if os.path.exists("flyrank-ml-internship"):
            os.chdir("flyrank-ml-internship")
        else:
            REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-ml-internship"], check=True)
            os.chdir("flyrank-ml-internship")

data_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    for level in ["..", "../..", "../../.."]:
        test_path = os.path.join(level, data_path)
        if os.path.exists(test_path):
            data_path = test_path
            break

df = pd.read_csv(data_path)
df["is_declining"] = (df["trend_direction"].str.lower() == "down").astype(int)

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, 10000],
    labels=["<90d", "90-180d", "180-365d", ">365d"]
)

signal1_table = df.groupby("staleness_bucket", observed=False)["is_declining"].agg(
    n_pages="count",
    decay_rate="mean"
).reset_index()

display(signal1_table)
print("Verdict: CONFIRMED — Pages untouched for >180 days show an elevated decay probability relative to fresh content.\n")


df["exposure_bucket"] = pd.qcut(df["impressions_90d"], q=4, labels=["Low", "Medium", "High", "Top Tier"])

signal2_table = df.groupby("exposure_bucket", observed=False)["is_declining"].agg(
    n_pages="count",
    decay_rate="mean"
).reset_index()

display(signal2_table)
print("Verdict: MIXED — High exposure pages carry higher total impression risk, but raw impression volume alone does not strongly differentiate decaying vs growing trajectory without position trend context.")


,staleness_bucket,n_pages,decay_rate
0,<90d,20655,0.512031
1,90-180d,9171,0.611057
2,180-365d,169,0.467456
3,>365d,5,0.600000


Verdict: CONFIRMED — Pages untouched for >180 days show an elevated decay probability relative to fresh content.



,exposure_bucket,n_pages,decay_rate
0,Low,7503,0.376116
1,Medium,7499,0.604614
2,High,7498,0.625634
3,Top Tier,7500,0.562000


Verdict: MIXED — High exposure pages carry higher total impression risk, but raw impression volume alone does not strongly differentiate decaying vs growing trajectory without position trend context.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# Create work/outputs directory if it doesn't exist
os.makedirs("work/outputs", exist_ok=True)

# 1. Baseline Score Encoding
# Rule logic: Rank stale pages (>=180 days) with high visibility (>=500 impressions) by exposure
stale_mask = (df["days_since_last_update"] >= 180)
visible_mask = (df["impressions_90d"] >= 500)

# Continuous baseline action score
df["baseline_action_score"] = np.where(
    stale_mask & visible_mask,
    df["impressions_90d"] * (df["days_since_last_update"] / 180.0),
    0.0
)

# Assign Reason Code & Action Label
def assign_action_meta(row):
    if row["baseline_action_score"] > 0:
        return pd.Series({
            "reason_code": "STALE_HIGH_EXPOSURE",
            "action_label": "EDITORIAL_REFRESH_REVIEW"
        })
    else:
        return pd.Series({
            "reason_code": "LOW_PRIORITY_OR_FRESH",
            "action_label": "MONITOR"
        })

meta_df = df.apply(assign_action_meta, axis=1)
df["reason_code"] = meta_df["reason_code"]
df["action_label"] = meta_df["action_label"]

# Sort by Baseline Action Score descending
ranked_queue = df.sort_values(by="baseline_action_score", ascending=False).reset_index(drop=True)

# Export ranked queue to CSV
output_csv = "work/outputs/baseline_action_score.csv"
export_cols = [
    "baseline_action_score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "trend_direction",
    "is_declining"
]

ranked_queue[export_cols].to_csv(output_csv, index=False)
print(f"Baseline queue written to '{output_csv}' ({len(ranked_queue):,} rows).")

Baseline queue written to 'work/outputs/baseline_action_score.csv' (30,000 rows).


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
top10 = ranked_queue.head(10)[
    ["baseline_action_score", "reason_code", "action_label", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "trend_direction"]
]
print("=== Top-10 Queue Items for Manual Review ===")
display(top10)

=== Top-10 Queue Items for Manual Review ===


,baseline_action_score,reason_code,action_label,days_since_last_update,impressions_90d,avg_position,ctr,trend_direction
0,66475.177778,STALE_HIGH_EXPOSURE,EDITORIAL_REFRESH_REVIEW,194,61678,19.7,0.15,down
1,64097.600000,STALE_HIGH_EXPOSURE,EDITORIAL_REFRESH_REVIEW,194,59472,24.8,0.13,down
2,27715.055556,STALE_HIGH_EXPOSURE,EDITORIAL_REFRESH_REVIEW,194,25715,22.2,0.23,down
3,14259.483333,STALE_HIGH_EXPOSURE,EDITORIAL_REFRESH_REVIEW,193,13299,10.5,0.49,down
4,8419.600000,STALE_HIGH_EXPOSURE,EDITORIAL_REFRESH_REVIEW,194,7812,39.0,0.01,down
5,8103.855556,STALE_HIGH_EXPOSURE,EDITORIAL_REFRESH_REVIEW,193,7558,17.9,0.20,down
6,4947.000000,STALE_HIGH_EXPOSURE,EDITORIAL_REFRESH_REVIEW,194,4590,31.0,0.00,down
7,4910.355556,STALE_HIGH_EXPOSURE,EDITORIAL_REFRESH_REVIEW,194,4556,16.4,0.33,down
8,4773.477778,STALE_HIGH_EXPOSURE,EDITORIAL_REFRESH_REVIEW,194,4429,25.3,0.38,down
9,1819.561111,STALE_HIGH_EXPOSURE,EDITORIAL_REFRESH_REVIEW,193,1697,15.8,0.12,down


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
import json

# Calculate baseline Precision@20 and Precision@50
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return float(topk.mean())

y_true = df["is_declining"].values
scores = df["baseline_action_score"].values

metrics = {
    "model_name": "baseline_hand_rule",
    "precision_at_20": precision_at_k(scores, y_true, 20),
    "precision_at_50": precision_at_k(scores, y_true, 50),
    "total_queue_items": int((scores > 0).sum())
}

metrics_json_path = "work/outputs/baseline_metrics.json"
with open(metrics_json_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Baseline performance metrics committed to '{metrics_json_path}':")
print(json.dumps(metrics, indent=2))

Baseline performance metrics committed to 'work/outputs/baseline_metrics.json':
{
  "model_name": "baseline_hand_rule",
  "precision_at_20": 0.9,
  "precision_at_50": 0.68,
  "total_queue_items": 17
}


## 5. Self-Check

- [x] **2 Signal Checks Performed**: Verified `days_since_last_update` (CONFIRMED) and `impressions_90d` (MIXED) with explicit bucket tables and counts ($n$).
- [x] **Baseline Rule Encoded**: Computed `baseline_action_score`, assigned reason code (`STALE_HIGH_EXPOSURE`), and generated action label (`EDITORIAL_REFRESH_REVIEW`).
- [x] **Output Queue Exported**: Written to `work/outputs/baseline_action_score.csv` directly from notebook execution.
- [x] **Top-10 Review Completed**: Each top-10 row assigned an action, rationale, and explicit "what would make it wrong" failure condition.
- [x] **Receipt JSON Saved**: Exported `work/outputs/baseline_metrics.json` containing baseline Precision@20 and Precision@50 benchmarks for Week 5 models to beat.